In [1]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("../data/raw")

players = pd.read_csv(RAW_DIR / "players.csv")
valuations = pd.read_csv(RAW_DIR / "player_valuations.csv")
appearances = pd.read_csv(RAW_DIR / "appearances.csv")
clubs = pd.read_csv(RAW_DIR / "clubs.csv")
competitions = pd.read_csv(RAW_DIR / "competitions.csv")

print("Players:", players.shape)
print("Valuations:", valuations.shape)
print("Appearances:", appearances.shape)
print("Clubs:", clubs.shape)
print("Competitions:", competitions.shape)

Players: (47702, 26)
Valuations: (616377, 6)
Appearances: (1862208, 13)
Clubs: (796, 17)
Competitions: (67, 11)


In [2]:
print("\nPLAYERS:\n",players.columns.tolist())
print("\nVALUATIONS:\n",players.columns.tolist())
print("\nAPPEARANCES:\n",players.columns.tolist())



PLAYERS:
 ['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']

VALUATIONS:
 ['player_id', 'first_name', 'last_name', 'name', 'last_season', 'current_club_id', 'player_code', 'country_of_birth', 'city_of_birth', 'country_of_citizenship', 'date_of_birth', 'sub_position', 'position', 'foot', 'height_in_cm', 'contract_expiration_date', 'agent_name', 'image_url', 'international_caps', 'international_goals', 'current_national_team_id', 'url', 'current_club_domestic_competition_id', 'current_club_name', 'market_value_in_eur', 'highest_market_value_in_eur']

APPEARANCES:
 

In [3]:
appearances.head(1)

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90


In [4]:
player_stats=appearances.groupby("player_id").agg({
    "minutes_played":"sum",
    "goals":"sum",
    "assists":"sum"
}).reset_index()

player_stats.head(10)

,player_id,minutes_played,goals,assists
0,10,8808,48,25
1,26,13508,0,0
2,65,8788,38,13
3,77,307,0,0
4,80,1080,0,0
5,109,3584,1,2
6,123,427,0,1
7,132,3987,9,4
8,215,6038,26,8
9,258,1075,0,2


In [5]:
valuations_sorted = valuations.sort_values("date", ascending=False)

latest_valuations = valuations_sorted.drop_duplicates(subset=["player_id"])

latest_valuations = latest_valuations[["player_id", "market_value_in_eur"]]

latest_valuations.head()

,player_id,market_value_in_eur
616376,1519157,100000
616284,668731,700000
616290,701224,2200000
616289,688019,800000
616288,685657,2000000


In [6]:
player_info = players[[
    "player_id",
    "name",
    "date_of_birth",
    "position",
    "current_club_id"
]]

player_info.head()

,player_id,name,date_of_birth,position,current_club_id
0,10,Miroslav Klose,1978-06-09 00:00:00,Attack,398
1,26,Roman Weidenfeller,1980-08-06 00:00:00,Goalkeeper,16
2,65,Dimitar Berbatov,1981-01-30 00:00:00,Attack,1091
3,77,Lúcio,1978-05-08 00:00:00,Defender,506
4,80,Tom Starke,1981-03-18 00:00:00,Goalkeeper,27


In [7]:
final_df=player_info.merge(player_stats,on='player_id',how='inner')
final_df=final_df.merge(latest_valuations,on='player_id',how='inner')
final_df.head()

,player_id,name,date_of_birth,position,current_club_id,minutes_played,goals,assists,market_value_in_eur
0,10,Miroslav Klose,1978-06-09 00:00:00,Attack,398,8808,48,25,1000000
1,26,Roman Weidenfeller,1980-08-06 00:00:00,Goalkeeper,16,13508,0,0,750000
2,65,Dimitar Berbatov,1981-01-30 00:00:00,Attack,1091,8788,38,13,1000000
3,77,Lúcio,1978-05-08 00:00:00,Defender,506,307,0,0,200000
4,80,Tom Starke,1981-03-18 00:00:00,Goalkeeper,27,1080,0,0,100000


In [8]:
final_df['date_of_birth']=pd.to_datetime(final_df['date_of_birth'])
final_df["age"]=(pd.Timestamp("today")-final_df["date_of_birth"]).dt.days


In [9]:
final_df["age"] = (pd.Timestamp("today") - final_df["date_of_birth"]).dt.days / 365
final_df = final_df.dropna(subset=["age"])
final_df["age"] = final_df["age"].astype(int)

In [10]:
final_df=final_df[final_df["minutes_played"]>0]

In [11]:
final_df["goals_per_90"] = final_df["goals"] / (final_df["minutes_played"] / 90)

final_df["assists_per_90"] = final_df["assists"] / (final_df["minutes_played"] / 90)

In [12]:
final_df["goal_contributions_per_90"] = (
    final_df["goals_per_90"] + final_df["assists_per_90"]
)

In [13]:
import numpy as np

final_df["log_market_value"] = np.log1p(final_df["market_value_in_eur"])

In [14]:
final_df[[
    "name",
    "age",
    "goals_per_90",
    "assists_per_90",
    "goal_contributions_per_90",
    "market_value_in_eur",
    "log_market_value"
]].head()

,name,age,goals_per_90,assists_per_90,goal_contributions_per_90,market_value_in_eur,log_market_value
0,Miroslav Klose,47,0.490463,0.255450,0.745913,1000000,13.815512
1,Roman Weidenfeller,45,0.000000,0.000000,0.000000,750000,13.527830
2,Dimitar Berbatov,45,0.389167,0.133136,0.522303,1000000,13.815512
3,Lúcio,48,0.000000,0.000000,0.000000,200000,12.206078
4,Tom Starke,45,0.000000,0.000000,0.000000,100000,11.512935


In [15]:
features_df = final_df[[
    "age",
    "goals_per_90",
    "assists_per_90",
    "goal_contributions_per_90",
    "position",
    "log_market_value"
]].copy()

features_df.head()

,age,goals_per_90,assists_per_90,goal_contributions_per_90,position,log_market_value
0,47,0.490463,0.255450,0.745913,Attack,13.815512
1,45,0.000000,0.000000,0.000000,Goalkeeper,13.527830
2,45,0.389167,0.133136,0.522303,Attack,13.815512
3,48,0.000000,0.000000,0.000000,Defender,12.206078
4,45,0.000000,0.000000,0.000000,Goalkeeper,11.512935


In [16]:
features_df = final_df[[
    "age",
    "goals_per_90",
    "assists_per_90",
    "goal_contributions_per_90",
    "position",
    "log_market_value"
]].copy()

features_df.head()

,age,goals_per_90,assists_per_90,goal_contributions_per_90,position,log_market_value
0,47,0.490463,0.255450,0.745913,Attack,13.815512
1,45,0.000000,0.000000,0.000000,Goalkeeper,13.527830
2,45,0.389167,0.133136,0.522303,Attack,13.815512
3,48,0.000000,0.000000,0.000000,Defender,12.206078
4,45,0.000000,0.000000,0.000000,Goalkeeper,11.512935


In [17]:
features_df = pd.get_dummies(features_df, columns=["position"], drop_first=True)


In [18]:
X = features_df.drop("log_market_value", axis=1)
y = features_df["log_market_value"]

print(X.shape)
print(y.shape)


(27620, 8)
(27620,)


In [19]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(
    X,y,test_size=0.2,random_state=42
)
print(X_train.shape,X_test.shape)

(22096, 8) (5524, 8)


In [20]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train, y_train)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for the `lsqr` solver.`tol` is set as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. This parameter has no effect when fittingon dense data... versionadded:: 1.7",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary ` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False


In [21]:
y_pred = model.predict(X_test)

In [22]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("MAE:", mae)
print("RMSE:", rmse)
print("R2 Score:", r2)

MAE: 1.1016654768912642
RMSE: 1.4339111242549436
R2 Score: 0.15992226824820643


In [23]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

rf_pred = rf_model.predict(X_test)

rf_mae = mean_absolute_error(y_test, rf_pred)
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
rf_r2 = r2_score(y_test, rf_pred)

print("Random Forest MAE:", rf_mae)
print("Random Forest RMSE:", rf_rmse)
print("Random Forest R2 Score:", rf_r2)

Random Forest MAE: 0.9849321994036965
Random Forest RMSE: 1.2642344000563517
Random Forest R2 Score: 0.3469743928803747


In [24]:
from sklearn.ensemble import GradientBoostingRegressor

gb_model = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model.fit(X_train, y_train)

gb_pred = gb_model.predict(X_test)

gb_mae = mean_absolute_error(y_test, gb_pred)
gb_rmse = np.sqrt(mean_squared_error(y_test, gb_pred))
gb_r2 = r2_score(y_test, gb_pred)

print("Gradient Boosting MAE:", gb_mae)
print("Gradient Boosting RMSE:", gb_rmse)
print("Gradient Boosting R2 Score:", gb_r2)

Gradient Boosting MAE: 0.9355078234305274
Gradient Boosting RMSE: 1.2038249374398522
Gradient Boosting R2 Score: 0.4078909814469396


In [25]:
features_df_v2 = final_df[[
    "age",
    "minutes_played",
    "goals",
    "assists",
    "goals_per_90",
    "assists_per_90",
    "goal_contributions_per_90",
    "position",
    "log_market_value"
]].copy()

features_df_v2.head()

,age,minutes_played,goals,assists,goals_per_90,assists_per_90,goal_contributions_per_90,position,log_market_value
0,47,8808,48,25,0.490463,0.255450,0.745913,Attack,13.815512
1,45,13508,0,0,0.000000,0.000000,0.000000,Goalkeeper,13.527830
2,45,8788,38,13,0.389167,0.133136,0.522303,Attack,13.815512
3,48,307,0,0,0.000000,0.000000,0.000000,Defender,12.206078
4,45,1080,0,0,0.000000,0.000000,0.000000,Goalkeeper,11.512935


In [26]:
features_df_v2 = pd.get_dummies(
    features_df_v2,
    columns=["position"],
    drop_first=True
)

features_df_v2.head()

,age,minutes_played,goals,assists,goals_per_90,assists_per_90,goal_contributions_per_90,log_market_value,position_Defender,position_Goalkeeper,position_Midfield,position_Missing
0,47,8808,48,25,0.490463,0.255450,0.745913,13.815512,False,False,False,False
1,45,13508,0,0,0.000000,0.000000,0.000000,13.527830,False,True,False,False
2,45,8788,38,13,0.389167,0.133136,0.522303,13.815512,False,False,False,False
3,48,307,0,0,0.000000,0.000000,0.000000,12.206078,True,False,False,False
4,45,1080,0,0,0.000000,0.000000,0.000000,11.512935,False,True,False,False


In [27]:
X_v2 = features_df_v2.drop("log_market_value", axis=1)
y_v2 = features_df_v2["log_market_value"]

print("X_v2 shape:", X_v2.shape)
print("y_v2 shape:", y_v2.shape)

X_v2 shape: (27620, 11)
y_v2 shape: (27620,)


In [28]:
from sklearn.model_selection import train_test_split

X_train_v2, X_test_v2, y_train_v2, y_test_v2 = train_test_split(
    X_v2,
    y_v2,
    test_size=0.2,
    random_state=42
)

print(X_train_v2.shape, X_test_v2.shape)

(22096, 11) (5524, 11)


In [29]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

gb_model_v2 = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model_v2.fit(X_train_v2, y_train_v2)

gb_pred_v2 = gb_model_v2.predict(X_test_v2)

gb_mae_v2 = mean_absolute_error(y_test_v2, gb_pred_v2)
gb_rmse_v2 = np.sqrt(mean_squared_error(y_test_v2, gb_pred_v2))
gb_r2_v2 = r2_score(y_test_v2, gb_pred_v2)

print("Gradient Boosting V2 MAE:", gb_mae_v2)
print("Gradient Boosting V2 RMSE:", gb_rmse_v2)
print("Gradient Boosting V2 R2 Score:", gb_r2_v2)

Gradient Boosting V2 MAE: 0.8086562828798686
Gradient Boosting V2 RMSE: 1.0332243821648868
Gradient Boosting V2 R2 Score: 0.5638214572896523


In [30]:
club_strength = final_df.groupby("current_club_id")["market_value_in_eur"].mean().reset_index()

club_strength = club_strength.rename(
    columns={"market_value_in_eur": "club_avg_market_value"}
)

final_df = final_df.merge(club_strength, on="current_club_id", how="left")

final_df["log_club_avg_market_value"] = np.log1p(final_df["club_avg_market_value"])

final_df[["name", "current_club_id", "club_avg_market_value", "log_club_avg_market_value"]].head()

,name,current_club_id,club_avg_market_value,log_club_avg_market_value
0,Miroslav Klose,398,3.366143e+06,15.029278
1,Roman Weidenfeller,16,1.164602e+07,16.270475
2,Dimitar Berbatov,1091,1.412349e+06,14.160766
3,Lúcio,506,1.175469e+07,16.279763
4,Tom Starke,27,2.908456e+07,17.185718


In [31]:
features_df_v3 = final_df[[
    "age",
    "minutes_played",
    "goals",
    "assists",
    "goals_per_90",
    "assists_per_90",
    "goal_contributions_per_90",
    "log_club_avg_market_value",
    "position",
    "log_market_value"
]].copy()

features_df_v3 = pd.get_dummies(
    features_df_v3,
    columns=["position"],
    drop_first=True
)

features_df_v3.head()

,age,minutes_played,goals,assists,goals_per_90,assists_per_90,goal_contributions_per_90,log_club_avg_market_value,log_market_value,position_Defender,position_Goalkeeper,position_Midfield,position_Missing
0,47,8808,48,25,0.490463,0.255450,0.745913,15.029278,13.815512,False,False,False,False
1,45,13508,0,0,0.000000,0.000000,0.000000,16.270475,13.527830,False,True,False,False
2,45,8788,38,13,0.389167,0.133136,0.522303,14.160766,13.815512,False,False,False,False
3,48,307,0,0,0.000000,0.000000,0.000000,16.279763,12.206078,True,False,False,False
4,45,1080,0,0,0.000000,0.000000,0.000000,17.185718,11.512935,False,True,False,False


In [32]:
X_v3 = features_df_v3.drop("log_market_value", axis=1)
y_v3 = features_df_v3["log_market_value"]

print("X_v3 shape:", X_v3.shape)
print("y_v3 shape:", y_v3.shape)

X_v3 shape: (27620, 12)
y_v3 shape: (27620,)


In [33]:
X_train_v3, X_test_v3, y_train_v3, y_test_v3 = train_test_split(
    X_v3,
    y_v3,
    test_size=0.2,
    random_state=42
)

gb_model_v3 = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42
)

gb_model_v3.fit(X_train_v3, y_train_v3)

gb_pred_v3 = gb_model_v3.predict(X_test_v3)

gb_mae_v3 = mean_absolute_error(y_test_v3, gb_pred_v3)
gb_rmse_v3 = np.sqrt(mean_squared_error(y_test_v3, gb_pred_v3))
gb_r2_v3 = r2_score(y_test_v3, gb_pred_v3)

print("Gradient Boosting V3 MAE:", gb_mae_v3)
print("Gradient Boosting V3 RMSE:", gb_rmse_v3)
print("Gradient Boosting V3 R2 Score:", gb_r2_v3)

Gradient Boosting V3 MAE: 0.6490435439644178
Gradient Boosting V3 RMSE: 0.841488999214051
Gradient Boosting V3 R2 Score: 0.7106843740967954


In [34]:
feature_importance = pd.DataFrame({
    "feature": X_v3.columns,
    "importance": gb_model_v3.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by="importance",
    ascending=False
)

feature_importance

,feature,importance
7,log_club_avg_market_value,0.452243
0,age,0.250620
1,minutes_played,0.243015
2,goals,0.023909
6,goal_contributions_per_90,0.014147
3,assists,0.007105
4,goals_per_90,0.005973
9,position_Goalkeeper,0.001815
5,assists_per_90,0.000949
10,position_Midfield,0.000181


In [35]:
results_df = final_df.loc[X_test_v3.index].copy()

results_df["predicted_log_market_value"] = gb_pred_v3

results_df["predicted_market_value_eur"] = np.expm1(
    results_df["predicted_log_market_value"]
)

results_df["actual_market_value_eur"] = results_df["market_value_in_eur"]

results_df["value_gap_eur"] = (
    results_df["predicted_market_value_eur"] - results_df["actual_market_value_eur"]
)

results_df[[
    "name",
    "age",
    "position",
    "actual_market_value_eur",
    "predicted_market_value_eur",
    "value_gap_eur"
]].head()

,name,age,position,actual_market_value_eur,predicted_market_value_eur,value_gap_eur
24781,Ismaël Doukouré,22,Defender,20000000,1.562763e+07,-4.372368e+06
4404,Egor Filipenko,38,Defender,150000,1.471996e+05,-2.800389e+03
25239,Steven Nsimba,29,Attack,800000,3.789428e+05,-4.210572e+05
21216,Marcus Lindberg,25,Midfield,700000,7.230435e+05,2.304351e+04
7763,Edin Visca,36,Attack,300000,1.170771e+06,8.707714e+05


In [36]:
undervalued_players = results_df[
    (results_df["age"] <= 25) &
    (results_df["minutes_played"] >= 900) &
    (results_df["value_gap_eur"] > 0) &
    (results_df["actual_market_value_eur"] >= 100000)
].copy()

undervalued_players = undervalued_players.sort_values(
    by="value_gap_eur",
    ascending=False
)

undervalued_players[[
    "name",
    "age",
    "position",
    "minutes_played",
    "goals",
    "assists",
    "actual_market_value_eur",
    "predicted_market_value_eur",
    "value_gap_eur"
]].head(20)

,name,age,position,minutes_played,goals,assists,actual_market_value_eur,predicted_market_value_eur,value_gap_eur
21365,Gonçalo Ramos,24,Attack,10117,73,23,35000000,7.478560e+07,3.978560e+07
24824,Roony Bardghji,20,Attack,2714,14,4,15000000,4.144691e+07,2.644691e+07
24546,Savinho,22,Attack,7569,17,27,40000000,6.259201e+07,2.259201e+07
23121,Issa Kaboré,24,Defender,7433,1,9,4000000,2.569802e+07,2.169802e+07
18536,Rodrygo,25,Attack,17216,70,58,50000000,7.032666e+07,2.032666e+07
21466,Kang-in Lee,25,Midfield,12859,25,29,28000000,4.698373e+07,1.898373e+07
21395,Santiago Gimenez,25,Attack,8714,72,19,20000000,3.453081e+07,1.453081e+07
26336,Artem Smolyakov,22,Defender,5466,4,3,1800000,1.353965e+07,1.173965e+07
21848,Rayan Aït-Nouri,24,Defender,14455,11,26,40000000,5.034061e+07,1.034061e+07
23354,David Møller Wolfe,24,Defender,7652,6,12,10000000,2.031802e+07,1.031802e+07


In [37]:
undervalued_players["scouting_score"] = (
    0.45 * undervalued_players["value_gap_eur"].rank(pct=True) +
    0.25 * undervalued_players["goal_contributions_per_90"].rank(pct=True) +
    0.20 * undervalued_players["minutes_played"].rank(pct=True) +
    0.10 * (1 - undervalued_players["age"].rank(pct=True))
)

undervalued_players = undervalued_players.sort_values(
    by="scouting_score",
    ascending=False
)

undervalued_players[[
    "name",
    "age",
    "position",
    "minutes_played",
    "goals",
    "assists",
    "goal_contributions_per_90",
    "actual_market_value_eur",
    "predicted_market_value_eur",
    "value_gap_eur",
    "scouting_score"
]].head(20)

,name,age,position,minutes_played,goals,assists,goal_contributions_per_90,actual_market_value_eur,predicted_market_value_eur,value_gap_eur,scouting_score
24546,Savinho,22,Attack,7569,17,27,0.523187,40000000,6.259201e+07,2.259201e+07,0.934948
21365,Gonçalo Ramos,24,Attack,10117,73,23,0.854008,35000000,7.478560e+07,3.978560e+07,0.930623
21440,Arsen Zakharyan,22,Midfield,9419,22,28,0.477758,10000000,1.737919e+07,7.379188e+06,0.913495
18536,Rodrygo,25,Attack,17216,70,58,0.669145,50000000,7.032666e+07,2.032666e+07,0.903633
21395,Santiago Gimenez,25,Attack,8714,72,19,0.939867,20000000,3.453081e+07,1.453081e+07,0.892388
22134,Sem Steijn,24,Midfield,9664,61,17,0.726407,11000000,1.696652e+07,5.966516e+06,0.891869
24824,Roony Bardghji,20,Attack,2714,14,4,0.596905,15000000,4.144691e+07,2.644691e+07,0.889965
26280,Clement Bischoff,20,Attack,2994,3,13,0.480962,5000000,1.376778e+07,8.767778e+06,0.859862
21466,Kang-in Lee,25,Midfield,12859,25,29,0.377945,28000000,4.698373e+07,1.898373e+07,0.857093
22145,Bjorn Meijer,23,Defender,7672,8,14,0.258081,3500000,1.349758e+07,9.997577e+06,0.842907


In [38]:
from pathlib import Path

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(PROCESSED_DIR / "model_predictions.csv", index=False)
undervalued_players.to_csv(PROCESSED_DIR / "undervalued_players.csv", index=False)
feature_importance.to_csv(PROCESSED_DIR / "feature_importance.csv", index=False)

model_comparison = pd.DataFrame({
    "model": [
        "Linear Regression",
        "Random Forest",
        "Gradient Boosting V1",
        "Gradient Boosting V2",
        "Gradient Boosting V3"
    ],
    "mae": [
        mae,
        rf_mae,
        gb_mae,
        gb_mae_v2,
        gb_mae_v3
    ],
    "rmse": [
        rmse,
        rf_rmse,
        gb_rmse,
        gb_rmse_v2,
        gb_rmse_v3
    ],
    "r2_score": [
        r2,
        rf_r2,
        gb_r2,
        gb_r2_v2,
        gb_r2_v3
    ]
})

model_comparison.to_csv(PROCESSED_DIR / "model_comparison.csv", index=False)

model_comparison

,model,mae,rmse,r2_score
0,Linear Regression,1.101665,1.433911,0.159922
1,Random Forest,0.984932,1.264234,0.346974
2,Gradient Boosting V1,0.935508,1.203825,0.407891
3,Gradient Boosting V2,0.808656,1.033224,0.563821
4,Gradient Boosting V3,0.649044,0.841489,0.710684
